# 🐾 Cat's Eye — AI Video Clipper (Google Colab Edition)

**Turn long videos and live streams into captioned 9:16 vertical shorts with AI.**

### ⚡ Google Drive Source of Truth
- All durable checkpoints, database state, rendered clips, and model caches live in `Google Drive / MyDrive / CatsEye`.
- If your Colab session times out or disconnects unexpectedly, **simply reconnect and run this notebook again**.
- Cat's Eye will automatically restore the latest verified database checkpoint and continue running any interrupted jobs.

In [ ]:
# Step 1: Mount Google Drive (Required for Durable Storage)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Unpack or Clone Cat's Eye & Install Dependencies
import os, shutil, zipfile

DRIVE_DIR = '/content/drive/MyDrive/CatsEye'
DRIVE_ZIP = f'{DRIVE_DIR}/cats-eye-colab.zip'
LOCAL_ZIP = '/content/cats-eye-colab.zip'
APP_DIR   = '/content/peakcut'

os.makedirs(DRIVE_DIR, exist_ok=True)

# If uploaded to /content, persist a copy to Google Drive so reconnects don't need re-upload
if os.path.exists(LOCAL_ZIP) and not os.path.exists(DRIVE_ZIP):
    shutil.copy2(LOCAL_ZIP, DRIVE_ZIP)

if not os.path.exists(f'{APP_DIR}/scripts/colab_bootstrap.py'):
    if os.path.exists(DRIVE_ZIP):
        print('📦 Extracting Cat\'s Eye from Google Drive bundle...')
        os.makedirs(APP_DIR, exist_ok=True)
        with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
            z.extractall(APP_DIR)
    else:
        print('📤 Upload colab/cats-eye-colab.zip from your local project folder (needed once; saved to Drive):')
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith('.zip'):
                shutil.copy2(fname, DRIVE_ZIP)
                os.makedirs(APP_DIR, exist_ok=True)
                with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
                    z.extractall(APP_DIR)
                break

%cd /content/peakcut

# Ensure ffmpeg is installed and install Python dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -r requirements.txt
print('✅ Cat\'s Eye is installed and ready.')

In [ ]:
# Step 3: Launch Cat's Eye & Open Web UI
import os
from google.colab.output import eval_js

# Set your custom owner password or use Colab Secrets (PEAKCUT_OWNER_PASSWORD)
os.environ.setdefault("PEAKCUT_OWNER_PASSWORD", "cats-eye-colab-2026")
os.environ["PEAKCUT_COLAB"] = "1"

# Official Google Colab Native Port Proxy Link
proxy_url = eval_js("google.colab.kernel.proxyPort(8765)")
print("=" * 60)
print("🐾 CLICK TO OPEN CAT'S EYE IN YOUR BROWSER:")
print(proxy_url)
print(f"🔑 Password: {os.environ['PEAKCUT_OWNER_PASSWORD']}")
print("=" * 60)

# Start Cat's Eye with automated lease acquisition and checkpoint recovery
!python scripts/colab_bootstrap.py

### ℹ️ Disconnection & Recovery FAQ
- **What happens if Colab disconnects?**
  Nothing is lost. The database is snapshotted every 30 seconds to Google Drive, and intra-stage chunks (transcripts, clips) are uploaded to Drive upon completion.
- **Can I run multiple notebooks at once?**
  Cat's Eye enforces a session lease file (`lease.json`) on your Google Drive. If an old session died, the new session will automatically take over after 3 minutes, or you can supply `--force-lease`.
- **Terms of Service Compliance:**
  We use Google Colab's native `google.colab.kernel.proxyPort(8765)`. It runs directly within Google's authenticated infrastructure without external public reverse tunnels (like cloudflared or ngrok), remaining 100% compliant with Colab free-tier rules.